In [ ]:
import os
from pathlib import Path
import pandas as pd

def check_dataset_structure(base_path):
    base = Path(base_path)
    report = []

    
    splits = ['train', 'val', 'test']
    for split in splits:
        img_dir = base / split / 'images'
        lbl_dir = base / split / 'labels'
        
        if img_dir.exists() and lbl_dir.exists():
            images = sorted([f.stem for f in img_dir.glob('*') if f.suffix.lower() in ['.jpg', '.jpeg', '.png']])
            labels = sorted([f.stem for f in lbl_dir.glob('*.txt')])
            
            
            mismatched_imgs = set(images) - set(labels)
            mismatched_lbls = set(labels) - set(images)
            
            report.append({
                'Split': split,
                'Total Images': len(images),
                'Total Labels': len(labels),
                'Mismatched Images': len(mismatched_imgs),
                'Mismatched Labels': len(mismatched_lbls)
            })
        else:
            print(f" تحذير: المجلد {split} غير موجود أو بنيته غير صحيحة.")
            
    df_report = pd.DataFrame(report)
    print("\n--- تقرير هيكلية الداتاسيت ---")
    print(df_report.to_string(index=False))
    return df_report
    
check_dataset_structure('/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

def analyze_class_distribution(base_path, class_names, split='train'):
    label_dir = Path(base_path) / split / 'labels'
    class_counts = Counter()
    total_boxes = 0
    
    for label_file in label_dir.glob('*.txt'):
        with open(label_file, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if parts:
                    class_id = int(parts[0])
                    class_counts[class_id] += 1
                    total_boxes += 1
                    
    
    data = []
    for cid, count in class_counts.items():
        name = class_names[cid] if cid < len(class_names) else f"Unknown ({cid})"
        data.append({'Class': name, 'Count': count, 'Percentage': (count / total_boxes) * 100})
        
    df_class = pd.DataFrame(data)
    
    
    plt.figure(figsize=(10, 5))
    sns.set_theme(style="whitegrid")
    ax = sns.barplot(x='Class', y='Count', data=df_class, palette='Oranges_r')
    
    
    for p in ax.patches:
        ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=11, fontweight='bold')
        
    plt.title(f'Distribution of Annotation Classes in ({split.capitalize()} Set)', fontsize=14, fontweight='bold')
    plt.xlabel('Class Name', fontsize=12)
    plt.ylabel('Total Bounding Boxes', fontsize=12)
    plt.show()
    
    print("\n--- تقرير توزيع الأصناف ---")
    print(df_class.to_string(index=False))
    return df_class


class_names = {0: 'Smoke', 1: 'Fire'}
analyze_class_distribution('/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data', class_names, split='train')

In [ ]:
import cv2
import random

def plot_random_samples(base_path, class_names, split='train', num_samples=3):
    base = Path(base_path)
    img_dir = base / split / 'images'
    lbl_dir = base / split / 'labels'
    
    images = list(img_dir.glob('*'))
    samples = random.sample(images, min(num_samples, len(images)))
    
    fig, axes = plt.subplots(1, num_samples, figsize=(18, 6))
    if num_samples == 1: axes = [axes]
        
    colors = {0: (0, 0, 255), 1: (255, 0, 0)} 
    
    for idx, img_path in enumerate(samples):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h_img, w_img, _ = img.shape
        
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    cid, x_c, y_c, w, h = map(float, line.strip().split())
                    
                    
                    x1 = int((x_c - w / 2) * w_img)
                    y1 = int((y_c - h / 2) * h_img)
                    x2 = int((x_c + w / 2) * w_img)
                    y2 = int((y_c + h / 2) * h_img)
                    
                    color = colors.get(int(cid), (0, 255, 0))
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                    
                    label_text = class_names.get(int(cid), str(int(cid)))
                    cv2.putText(img, label_text, (x1, max(y1 - 10, 15)), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
                    
        axes[idx].imshow(img)
        axes[idx].set_title(f"{img_path.name}\nSize: {w_img}x{h_img}")
        axes[idx].axis('off')
        
    plt.tight_layout()
    plt.show()

# الاستخدام:
plot_random_samples('/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data', class_names, split='train', num_samples=3)

In [ ]:
def analyze_box_dimensions(base_path, split='train'):
    label_dir = Path(base_path) / split / 'labels'
    widths = []
    heights = []
    
    for label_file in label_dir.glob('*.txt'):
        with open(label_file, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) == 5:
                
                    widths.append(float(parts[3]))
                    heights.append(float(parts[4]))
                    
    df_boxes = pd.DataFrame({'Width': widths, 'Height': heights})
    df_boxes['Aspect_Ratio'] = df_boxes['Width'] / (df_boxes['Height'] + 1e-6)
    
    
    plt.figure(figsize=(8, 7))
    sns.jointplot(x='Width', y='Height', data=df_boxes, kind='hex', color='red')
    plt.suptitle('Bounding Box Bivariate Distribution (Width vs Height)', y=1.02, fontsize=12, fontweight='bold')
    plt.show()
    
    # حساب النِسَب الإحصائية الهامة أكاديمياً
    print("\n--- الإحصاءات الوصفية لأبعاد الكائنات ---")
    print(df_boxes.describe())
    
    small_objects = df_boxes[(df_boxes['Width'] < 0.1) & (df_boxes['Height'] < 0.1)]
    print(f"\n نسبة الكائنات الصغيرة جداً (أقل من 10% من حجم الصورة): {len(small_objects)/len(df_boxes)*100:.2f}%")
    
    return df_boxes

# الاستخدام:
analyze_box_dimensions('/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data', split='train')

In [ ]:

!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.0 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires nu

In [ ]:
import os

def find_dataset_root():
    print("=== بدء فحص مجلدات المدخلات في Kaggle ===")
    input_root = '/kaggle/input/'
    
    
    base_folders = os.listdir(input_root)
    print(f"📁 المجلدات الرئيسية الموجودة: {base_folders}\n")
    
    
    detected_path = None
    for root, dirs, files in os.walk(input_root):
        
        if 'train' in dirs and ('val' in dirs or 'valid' in dirs):
            detected_path = root
            print(f"🎯 تم العثور على مجلد البيانات الحقيقي: {detected_path}")
            print(f"   المحتويات داخله: {os.listdir(detected_path)}")
            break
            
    if not detected_path:
        print("❌ لم يتم العثور على بنية YOLO التقليدية (train/val) تلقائياً.")
        print("إليك الهيكلية الكاملة للمجلدات")
        for root, dirs, files in os.walk(input_root):
            level = root.replace(input_root, '').count(os.sep)
            indent = ' ' * 4 * (level)
            print(f"{indent}📂 {os.path.basename(root)}/")
            
    return detected_path


REAL_BASE_PATH = find_dataset_root()

=== بدء فحص مجلدات المدخلات في Kaggle ===
📁 المجلدات الرئيسية الموجودة: ['datasets']

🎯 تم العثور على مجلد البيانات الحقيقي: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data
   المحتويات داخله: ['val', 'test', 'train']


In [ ]:
import yaml

if REAL_BASE_PATH:
    
    data_config = {
        'path': REAL_BASE_PATH,
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images' if os.path.exists(os.path.join(REAL_BASE_PATH, 'test')) else 'val/images',
        'names': {
            1: 'Fire',
            0: 'Smoke'
        }
    }

    with open('/kaggle/working/data.yaml', 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False)
    print("✅ تم إنشاء ملف data.yaml بنجاح بالمسار الحقيقي!")
else:
    print("يرجى مراجعة مخرجات الفحص أولاً.")

✅ تم إنشاء ملف data.yaml بنجاح بالمسار الحقيقي!


In [ ]:
from ultralytics import YOLO


model = YOLO('yolo26m.pt') 

metrics = model.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=640,
    batch=32,
    device=[-1,-1],
    mosaic=1.0,             # موجود شرحها على موقع التراليتيكس
    close_mosaic=20,
    degrees=10.0,
     hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mixup=0.1, 
    workers=4,
    save=True,
    plots=True
)

In [ ]:
import yaml
from ultralytics import YOLO

original_yaml_path = (
    "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data.yaml"
)


with open(original_yaml_path, "r") as f:
  data = yaml.safe_load(f)


data["path"] = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo"


fixed_yaml_path = "data.yaml"
with open(fixed_yaml_path, "w") as f:
  yaml.dump(data, f)


model = YOLO("/kaggle/input/datasets/issahasan43/yolov26m-wheights/best.pt")
metrics = model.val(data=fixed_yaml_path)

Ultralytics 8.4.111 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26m summary (fused): 132 layers, 20,350,994 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 375.7±238.4 MB/s, size: 193.6 KB)
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels... 3094 images, 1375 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 3099/3099 1.1Kit/s 2.9s<0.1s
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-det

In [ ]:
import shutil


shutil.make_archive(
    "/kaggle/working/val_output_final", "zip", "/kaggle/working/runs/detect/val-6"
)

'/kaggle/working/val_output_final.zip'